# Assignment 2 – RAG
**Students:** Ilay Nisim & Boris Kogan  
**Due date:** 28.4.26

---

## Setup
Install all packages needed for the entire assignment (Tasks 1–7 + Bonus).

In [ ]:
import sys
!{sys.executable} -m pip install -q openai pandas openpyxl datasets langchain langchain-community faiss-cpu sentence-transformers pypdf cryptography ragas python-dotenv


In [2]:
import os
import json
import pandas as pd
from openai import OpenAI
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current directory

NEBIUS_API_KEY   = os.environ["NEBIUS_API_KEY"]
NEBIUS_BASE_URL  = "https://api.studio.nebius.ai/v1/"

LLM_MODEL        = "meta-llama/Llama-3.3-70B-Instruct"   # generation (Tasks 1, 4, 5)
JUDGE_MODEL      = "deepseek-ai/DeepSeek-V3-0324"         # judge (Tasks 6, 7)
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"               # embeddings (Tasks 3–7)

client = OpenAI(api_key=NEBIUS_API_KEY, base_url=NEBIUS_BASE_URL)

def llm_generate(question: str, system_prompt: str = None, model: str = LLM_MODEL) -> str:
    """Single reusable helper for all LLM calls in this assignment."""
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.1,
        max_tokens=500,
    )
    return response.choices[0].message.content.strip()

print("Setup complete. Model:", LLM_MODEL)

/Users/YOUR_USER/dev/cloud/nebius/lesson5_rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete. Model: meta-llama/Llama-3.3-70B-Instruct


---
## Dataset preparation

Load [FinanceBench](https://huggingface.co/datasets/PatronusAI/financebench), drop `metrics-generated` questions, fix dead doc links, and parse the `evidence` column into flat fields reused across Tasks 3–7.

In [3]:
# Load
df = load_dataset("PatronusAI/financebench", split="train").to_pandas()

# Drop metrics-generated
df = df[df["question_type"] != "metrics-generated"].copy()

# Rename answer → ground_truth
df = df.rename(columns={"answer": "ground_truth"})

# Fix dead doc_link URLs
df["doc_link"] = df["doc_name"].apply(
    lambda name: f"https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs/{name}.pdf"
)

# Parse evidence column → evidence_page_nums (list[int]) + evidence_texts (list[str])
# The column can be: a JSON string, a list of dicts, or a numpy array of dicts
import numpy as np

def parse_evidence(ev):
    if isinstance(ev, str):
        ev = json.loads(ev)
    if isinstance(ev, np.ndarray):
        ev = ev.tolist()
    if not isinstance(ev, list):
        ev = [ev]
    page_nums, texts = [], []
    for e in ev:
        if isinstance(e, dict):
            page_nums.append(int(e.get("evidence_page_num", -1)))
            texts.append(str(e.get("evidence_text", "")))
        else:
            # numpy structured array row or similar — convert to dict first
            e = dict(e)
            page_nums.append(int(e.get("evidence_page_num", -1)))
            texts.append(str(e.get("evidence_text", "")))
    return page_nums, texts

parsed = df["evidence"].apply(parse_evidence)
df["evidence_page_nums"] = parsed.apply(lambda x: x[0])
df["evidence_texts"]     = parsed.apply(lambda x: x[1])

# Doc metadata lookup: doc_name → {company, doc_period, doc_link}  (used by Task 3)
doc_meta = (
    df[["doc_name", "company", "doc_period", "doc_link"]]
    .drop_duplicates(subset="doc_name")
    .set_index("doc_name")
    .to_dict(orient="index")
)

# Sort
df = df.sort_values("financebench_id").reset_index(drop=True)

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"Rows: {len(df)}  |  Unique docs: {len(doc_meta)}")
print(df["question_type"].value_counts().to_string())
print()
print(df[["financebench_id", "question_type", "question", "doc_name"]].head(3).to_string(index=False))

Rows: 100  |  Unique docs: 42
question_type
domain-relevant    50
novel-generated    50

      financebench_id   question_type                                                                                                                                                                                      question                    doc_name
financebench_id_00005 domain-relevant              Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.            CORNING_2022_10K
financebench_id_00070 domain-relevant Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why. AMERICANWATERWORKS_2022_10K
financebench_id_00080 domain-relevant               Does Paypal have positive working capital based on FY2022 data? If working capital is not a

---
## Task 1 – naive generation (10 points)

We send the first 5 questions of each type straight into `Llama-3.3-70B-Instruct` — no retrieval, no grounding — and compare the outputs to the ground truth.

In [4]:
# First 5 per type — kept as separate named slices for reuse in Tasks 5, 6, 7
task1_domain = df[df["question_type"] == "domain-relevant"].head(5).copy()
task1_novel  = df[df["question_type"] == "novel-generated"].head(5).copy()
task1_df     = pd.concat([task1_domain, task1_novel]).sort_values("financebench_id").copy()

print(f"{len(task1_df)} questions selected  ({len(task1_domain)} domain-relevant, {len(task1_novel)} novel-generated):")
for _, row in task1_df.head(3).iterrows():
    print(f"  [{row['financebench_id']}] ({row['question_type']}) {row['question'][:120]}")
print("  ...")

10 questions selected  (5 domain-relevant, 5 novel-generated):
  [financebench_id_00005] (domain-relevant) Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric f
  [financebench_id_00070] (domain-relevant) Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or rele
  [financebench_id_00080] (domain-relevant) Does Paypal have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric fo
  ...


In [5]:
NAIVE_SYSTEM_PROMPT = (
    "You are a helpful financial analyst. "
    "Answer the question to the best of your knowledge. "
    "Be concise."
)

naive_answers = []
for i, (_, row) in enumerate(task1_df.iterrows()):
    if i < 3:
        print(f"[{row['financebench_id']}] querying...")
    answer = llm_generate(row["question"], system_prompt=NAIVE_SYSTEM_PROMPT)
    naive_answers.append(answer)
    if i < 3:
        print(f"  → {answer[:150]}\n")

task1_df["naive_answer"] = naive_answers
print("Done.")

[financebench_id_00005] querying...
  → Based on Corning's FY2022 data, the company's current assets were $7.85 billion and current liabilities were $4.93 billion. This results in a positive

[financebench_id_00070] querying...
  → Based on American Water Works' FY2022 data, the company's current assets were approximately $2.43 billion, and its current liabilities were around $2.

[financebench_id_00080] querying...
  → Based on PayPal's FY2022 data, the company's working capital is negative. As of December 31, 2022, PayPal's current assets were $63.8 billion, and its

Done.


In [6]:
# Sample results for manual review
pd.set_option("display.max_colwidth", 250)
task1_df[["financebench_id", "question_type", "question", "naive_answer", "ground_truth"]].head(2)

,financebench_id,question_type,question,naive_answer,ground_truth
0,financebench_id_00005,domain-relevant,"Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.","Based on Corning's FY2022 data, the company's current assets were $7.85 billion and current liabilities were $4.93 billion. This results in a positive working capital of $2.92 billion.\n\nWorking capital is a relevant metric for Corning, as it is...",Yes. Corning had a positive working capital amount of $831 million by FY 2022 close. This answer considers only operating current assets and current liabilities that were clearly shown in the balance sheet.
1,financebench_id_00070,domain-relevant,"Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.","Based on American Water Works' FY2022 data, the company's current assets were approximately $2.43 billion, and its current liabilities were around $2.55 billion. This results in negative working capital of approximately -$120 million.\n\nHowever,...","No, American Water Works had negative working capital of -$1561M in FY 2022."


In [7]:
# Heuristic pre-fill — update verdicts manually after reviewing the table above
REFUSE_PHRASES = [
    "i don't have", "i cannot", "i don't know", "i'm unable",
    "i need more", "please provide", "could you clarify",
    "without access", "not available to me",
]

task1_df["verdict"] = task1_df["naive_answer"].apply(
    lambda a: "refused" if any(p in a.lower() for p in REFUSE_PHRASES) else "wrong"
)

# ── Manual corrections ────────────────────────────────────────────────────────
# Allowed values: correct | partially correct | wrong | refused
# task1_df.loc[task1_df.financebench_id == "financebench_id_0", "verdict"] = "correct"

In [8]:
OUT_FILE = "assignment2_naive_generation_ilay_nisim_boris_kogan.xlsx"
task1_df[
    ["financebench_id", "question_type", "question", "naive_answer", "ground_truth", "verdict"]
].to_excel(OUT_FILE, index=False)
print(f"Saved → {OUT_FILE}")

Saved → assignment2_naive_generation_ilay_nisim_boris_kogan.xlsx


### Task 1 – Discussion

**1. Cases where the model refused or asked for more information - why?**  
Only 1 refusal: `financebench_id_00288` (cash drop FY2023 → Q2 FY2024). The model said it lacks access to real-time data — Looks like the reason for the refusal is that Q2 FY2024 falls beyond the model's training cutoff. 

**2. Cases where the model answered confidently - spot-check against the ground truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?**  
Cases where the model answered confidently divide into **2 correct**, **5 partially correct**, **2 hallucinated**:
- **Correct:** `00206` (JPM gross margin not relevant for a bank — probably because it requires pure financial reasoning, domain-relevant) and `00382` (MGM Las Vegas highest EBITDAR — well-known fact).
- **Partially correct:** right direction or entity but wrong numbers. E.g. `00005` Corning WC positive ✓ but $3.02B vs $831M (model used all current items; ground truth uses only *operating* items); `00299` correct segment (Corporate) but reported +$254M instead of −$473M (model dropped the negative sign).
- **Hallucinated:** `00080` PayPal WC (wrong sign, fabricated $63.8B / $83.8B figures wrapped in convincing payment-float reasoning); `00283` Pfizer–Upjohn spin-off cost ($12B vs actual $77.78M — 150× off, zero hedging). These are the most dangerous failures — authoritative tone, wrong facts.

**3. Are there patterns by *question_type*? Do some types fail more than others?**  
- **domain-relevant (5):** 1 correct / 3 partial / 1 wrong.
- **novel-generated (5):** 1 correct / 2 partial / 1 wrong / 1 refused.

The model succeeds when the answer needs *concepts* or *widely-known facts* and fails when it needs *exact figures from a specific filing*. This is exactly the gap RAG should close — expect the largest improvement on novel-generated questions and on the working-capital triad (00005/00070/00080), where without the filing the model can't apply the narrower *operating* WC definition.

---
## Task 2 – RAG reminder (5 points)

### 1. Indexing (documents → chunk + embed → vector store)

**How does it contribute to the pipeline?**  

Indexing is the most important part, it is the "ground truth" of how we interpreted meaning and it will affect all future search. it includes chunking which splits documents into pieces, embedding converts text to vectors for semantic similarity, and the vector store enables fast lookup. since it is done once, may take a lot of time, and need to be optimized, improving it is much more complex. 

**Where can it fail, concrete examples?**  
It can decide not to embed rare words, it can fail in close meanings, it can fail if data is updated.
Chunking can fail in overlaps for example a financial table split across two chunks loses its row context; a term like "EBITDAR" may be poorly embedded by a general-purpose model; a stale index after a document update.

**Once or per query?**  

It is done once and is the base for all search and retrieval this is why it is so important.

### 2. Retrieval (user query → retrieval)

**How does it contribute to the pipeline?**  

Retrieval is getting the data, it is also critical since it should bring the right amount of documents not to overload the context with irrelevant data but enough to give a full answer. the quality and relevance of the data fetched is critical if this fails the generation would fail.
**Where can it fail, concrete examples?**  
Retrieval fails when it fetches from the wrong document, returns wrong-k results, or
scores irrelevant chunks higher. When not bringing enough answers, for example when calculating capital gain, giving partial information then sums calculation would be wrong, giving irrelevant data.  

**Once or per query?**  

Per query.

### 3. Generation (generation)

**How does it contribute to the pipeline?**  

Generation converts retrieved chunks + query into a coherent answer for the user;  Depending on the data retrieved, llm should compose a reasonable answer. relevant if it had the data and saying it doesn't know if it doesn't have it.

**Where can it fail?**  

It can miss some of the data, it can add irrelevant other data. without strong instruction it may hallucinate beyond the context or ignore retrieved facts. For example the model may attribute a number to the wrong company when two filings are in context; it can ignore a retrieved passage saying "not applicable" and  hallucinates a figure anyway.      
The model should use the correct information from the chunks, know when there is not enough information. Even of all the data is there, it might not use it correctly. it should, given the relevant data, compose the correct answer. 
**Once or per query?**

Per query.

---
## Task 3 – embed documents (15 points)

Downloading PDFs, splitting embedding and persisting a FAISS index to disk (`faiss_chunk1000/`) for reuse across Tasks 4–7.

In [ ]:
# Download the PDFs for every doc_name in the filtered dataset.
# Cached under ./pdfs/ so re-running is a no-op.
#
# NOTE: pypdf needs `cryptography` (or `pycryptodome`) to decrypt AES-encrypted filings.
# If you see "DependencyError: cryptography>=3.1 is required for AES algorithm" after
# running this cell once, restart the kernel (Kernel → Restart) and re-run — pypdf
# caches the missing-crypto state from an earlier import.
import sys
!{sys.executable} -m pip install -q --upgrade pypdf
!{sys.executable} -m pip install -q cryptography pycryptodome

from pathlib import Path
import urllib.request
import urllib.error

PDF_DIR = Path("pdfs")
PDF_DIR.mkdir(exist_ok=True)
# github.com/.../tree/main/pdfs URL is the browsable directory page — it returns the HTML of GitHub's file
# listing UI, not the PDF bytes. raw.githubusercontent.com/patronus-ai/financebench/main/pdfs/<name>.pdf is the
#direct raw download URL 
GITHUB_PDF_BASE = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"

needed_docs = sorted(df["doc_name"].unique())
print(f"Need {len(needed_docs)} PDFs")

missing = []
for name in needed_docs:
    dest = PDF_DIR / f"{name}.pdf"
    if dest.exists() and dest.stat().st_size > 0:
        continue
    url = f"{GITHUB_PDF_BASE}/{name}.pdf"
    try:
        urllib.request.urlretrieve(url, dest)
    except urllib.error.HTTPError as e:
        missing.append((name, e.code))
        if dest.exists():
            dest.unlink()

have = sum(1 for n in needed_docs if (PDF_DIR / f"{n}.pdf").exists())
print(f"Downloaded/cached: {have}/{len(needed_docs)}")
if missing:
    print(f"Missing (HTTP errors): {missing}")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Need 42 PDFs
Downloaded/cached: 42/42


In [15]:
# Load each PDF page-by-page with PyPDFLoader and attach standardized metadata.
# PyPDFLoader's default `page` metadata is already 0-indexed → matches evidence_page_num.
from langchain_community.document_loaders import PyPDFLoader

docs_per_page = []
for name in needed_docs:
    path = PDF_DIR / f"{name}.pdf"
    if not path.exists():
        continue
    meta = doc_meta[name]
    pages = PyPDFLoader(str(path)).load()   # one Document per page
    for p in pages:
        page_num = p.metadata.get("page", 0)
        p.metadata = {
            "doc_name":    name,
            "company":     meta["company"],
            "doc_period":  meta["doc_period"],
            "page_number": int(page_num),
        }
        docs_per_page.append(p)

print(f"Loaded {len(docs_per_page)} page-level Documents from {len(needed_docs)} PDFs")
print("Sample metadata:", docs_per_page[0].metadata)

Loaded 5448 page-level Documents from 42 PDFs
Sample metadata: {'doc_name': '3M_2022_10K', 'company': '3M', 'doc_period': 2022, 'page_number': 0}


In [16]:
# Split into chunks; LangChain propagates parent page metadata to each chunk automatically.
# In recent langchain versions RecursiveCharacterTextSplitter lives in its own package.
import sys
!{sys.executable} -m pip install -q langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(docs_per_page)

print(f"Split {len(docs_per_page)} pages → {len(chunks)} chunks")
print("Sample chunk metadata:", chunks[0].metadata)
print("Sample chunk text (first 200 chars):")
print(chunks[0].page_content[:200])


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Split 5448 pages → 24218 chunks
Sample chunk metadata: {'doc_name': '3M_2022_10K', 'company': '3M', 'doc_period': 2022, 'page_number': 0}
Sample chunk text (first 200 chars):
T able of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 
10-K
 
 
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the f


In [17]:
# Embed with BAAI/bge-small-en-v1.5 and build/load the FAISS index on disk.
# Folder name includes the chunk size so Task 7 can add `faiss_chunk500`, `faiss_chunk2000`, etc.
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

FAISS_DIR = "faiss_chunk1000"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

if Path(FAISS_DIR).exists():
    vectorstore = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)
    print(f"Loaded existing FAISS index from {FAISS_DIR}")
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(FAISS_DIR)
    print(f"Built and saved FAISS index → {FAISS_DIR}")

print(f"Total vectors in store: {vectorstore.index.ntotal}")

/var/folders/lg/vvqm3z4111l_5k_q7hn8kcmc0000gn/T/ipykernel_48557/638132544.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14910.88it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Built and saved FAISS index → faiss_chunk1000
Total vectors in store: 24218


In [20]:
# Retrieval sanity check on 3 questions: right doc? right page? evidence text visible in chunks?
#
# Evidence-text visibility is measured by word 8-gram overlap between each chunk and
# the union of the question's evidence_texts. Any shared 8-gram ⇒ evidence text is
# verbatim-present in that chunk.  (Shorter evidence texts fall back to smaller n.)
# The original verbose printout is preserved below for manual inspection.

SAMPLE_IDS = ["financebench_id_00005", "financebench_id_00080", "financebench_id_00206"]
K = 5

def _shingles(text, n=8):
    words = str(text).lower().split()
    n = min(n, max(1, len(words) - 1))
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}

def evidence_overlap(chunk_text, evidence_texts, n=8):
    """Count of shared n-gram shingles, and total evidence shingles (for normalization)."""
    chunk_shings = _shingles(chunk_text, n=n)
    ev_shings = set()
    for ev in evidence_texts:
        ev_shings |= _shingles(ev, n=n)
    return len(chunk_shings & ev_shings), len(ev_shings)

sample_df = df[df["financebench_id"].isin(SAMPLE_IDS)].copy()
md_lines = ["# Task 3 – Retrieval Sanity Check (3 Questions)", ""]

for _, row in sample_df.iterrows():
    q              = row["question"]
    expected_doc   = row["doc_name"]
    expected_pages = row["evidence_page_nums"]
    expected_texts = row["evidence_texts"]

    retrieved = vectorstore.similarity_search(q, k=K)

    doc_hits = sum(1 for c in retrieved if c.metadata["doc_name"] == expected_doc)
    page_hit = any(
        c.metadata["doc_name"] == expected_doc
        and c.metadata["page_number"] in expected_pages
        for c in retrieved
    )
    text_hit = any(
        evidence_overlap(c.page_content, expected_texts)[0] > 0 for c in retrieved
    )

    # ── console: verbose, manual-inspection friendly ─────────────────────────
    print("=" * 95)
    print(f"[{row['financebench_id']}]  {q[:110]}")
    print(f"Expected doc: {expected_doc}  |  evidence pages: {expected_pages}")
    print(f"Chunks from correct doc: {doc_hits}/{K}   |   Any chunk on evidence page: {page_hit}   |   Evidence text visible (8-gram): {text_hit}")
    for i, c in enumerate(retrieved, 1):
        page_ok = (c.metadata["doc_name"] == expected_doc
                   and c.metadata["page_number"] in expected_pages)
        ov_hits, ov_total = evidence_overlap(c.page_content, expected_texts)
        marker = "✓" if page_ok else " "
        text_marker = "★" if ov_hits > 0 else " "
        snippet = c.page_content[:120].replace("\n", " ")
        print(f"  {marker}{text_marker} rank {i}  {c.metadata['doc_name']}  p{c.metadata['page_number']}  "
              f"[ev 8-grams: {ov_hits}/{ov_total}]  | {snippet}...")
    if expected_texts:
        print(f"  Evidence text (truncated): {expected_texts[0][:220]}...")

    # ── markdown: structured capture for task3_observe_3docs.md ──────────────
    md_lines += [
        f"## {row['financebench_id']}",
        "",
        f"**Question:** {q}",
        "",
        f"- Expected doc: `{expected_doc}`",
        f"- Evidence pages: `{expected_pages}`",
        f"- Chunks from correct doc: **{doc_hits}/{K}**",
        f"- Any chunk on an evidence page: **{page_hit}**",
        f"- Evidence text visible (8-gram overlap): **{text_hit}**",
        "",
        "| rank | doc | page | page_match | ev_8grams | chunk (first 160 chars) |",
        "|------|-----|------|------------|-----------|--------------------------|",
    ]
    for i, c in enumerate(retrieved, 1):
        page_ok = (c.metadata["doc_name"] == expected_doc
                   and c.metadata["page_number"] in expected_pages)
        ov_hits, ov_total = evidence_overlap(c.page_content, expected_texts)
        snippet = c.page_content[:160].replace("\n", " ").replace("|", "\\|")
        md_lines.append(
            f"| {i} | `{c.metadata['doc_name']}` | {c.metadata['page_number']} | "
            f"{'✓' if page_ok else ' '} | {ov_hits}/{ov_total} | {snippet}... |"
        )
    md_lines += [""]
    for j, ev in enumerate(expected_texts):
        md_lines += [f"**Evidence text [{j}]** (page {expected_pages[j] if j < len(expected_pages) else '?'}):", "", "> " + str(ev).replace("\n", " "), ""]

out_path = Path("task3_observe_3docs.md")
out_path.write_text("\n".join(md_lines), encoding="utf-8")
print(f"\nMarkdown report written → {out_path}")

[financebench_id_00005]  Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevan
Expected doc: CORNING_2022_10K  |  evidence pages: [59]
Chunks from correct doc: 3/5   |   Any chunk on evidence page: False   |   Evidence text visible (8-gram): False
     rank 1  CORNING_2022_10K  p101  [ev 8-grams: 0/171]  | (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Heml...
     rank 2  CORNING_2022_10K  p102  [ev 8-grams: 0/171]  | (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Heml...
     rank 3  3M_2022_10K  p37  [ev 8-grams: 0/171]  | not defined under U.S. generally accepted accounting principles and may not be computed the same as similarly titled mea...
     rank 4  3M_2023Q2_10Q  p70  [ev 8-grams: 0/171]  | Current assets $ 15,754 $ 14,688 $ 1,066  Less: Current liabilities 10,936 9,523 1,413  Worki

### Task 3 – Observations

**Right document (doc_name match)?**  
For Corning(id_00005), 3/5 right docs. ranks 1-2, 5 were the right documents, rank 3-4 were another company(3M).
For PayPal(id_00080) all 5 ranks were from correct doc.
For JPM, none of the ranks were from the correct doc rank2 was the only one from JPMORGAN_2022Q2_10Q instead of JPMORGAN_2022_10K.

**Evidence text visible in retrieved chunks?**  
trying to find 8-grams (a contiguous sequence of 8 words, in order) in retrieved text in all 3 companies resulted in zero match.

**Right page (chunk page_number vs evidence_page_num)?**  
In all the companies retrieved chunks were not found in evidence page.
For Corning and PayPal retrieval lands on narrative/MD&A pages that
talk about working capital, not the balance-sheet table that holds the
numbers. For JPM it misses the 10-K entirely because "gross margins"
isn't vocabulary JPM's filing uses. A reranker or hybrid retriever might help later.

To summarize results looks pretty poor, in total 8/15 documents were correct, no text in evidence text, no page number that matches evidence pages number. This may be since as explained FinanceBench is a hard dataset and BGE-small is a general-purpose embedder.